# Agente revisor de rendiciones de gastos — Tarea Final Curso 1 (Grupo 05)

Este notebook **solo importa** desde `src/`: la lógica vive en los módulos.
Ejecútalo de arriba abajo desde la carpeta `tarea-final-c1/`. Todos los datos son ficticios.

## 0. Ficha de reproducción

Modelo, temperatura, versiones de librerías, variables de entorno y datos usados.
La tabla completa está en `README.md`. Esta celda muestra lo que realmente está instalado.

In [ ]:
from importlib.metadata import version

from src.config import MODELO_DEL_CURSO, TEMPERATURA

print('Modelo:', MODELO_DEL_CURSO, '| temperatura:', TEMPERATURA)
for libreria in ['langgraph', 'langchain', 'langchain-google-genai', 'langchain-redis', 'redis', 'redisvl']:
    print(f'{libreria}=={version(libreria)}')

## 1. Caso y criterio de éxito

**Problema.** En **Comercial Los Alerces SpA** (empresa ficticia), cada colaborador elige la política y la categoría de sus gastos. Es común que elija mal, porque una misma categoría (peaje, combustible, almuerzo) existe en varias políticas, y cada una tiene un tratamiento contable distinto. Clasificar mal genera cuentas equivocadas, reportes distorsionados y reprocesos.

**Usuario.** El analista de Control de Gestión que hace la primera revisión normativa. El colaborador que rinde recibe la observación y corrige.

**Entrada.** Un gasto rendido: fechas, monto, tipo de documento, glosa, política y categoría declaradas, unidad de negocio, fondo y campos obligatorios. También acepta preguntas sobre la política. Lo declarado es una pista, no la respuesta.

**Alcance.** Revisa gastos de a uno contra la política POL-RG-01 v1.6. Puede buscar en la política (RAG), validar reglas exactas, emitir un veredicto y registrarlo en una base de prueba. No paga, no borra registros, no aprueba sin validar y no responde temas ajenos a las rendiciones.

**Salida esperada.** Una decisión (aprobar, observar o rechazar), con la política, categoría y cuenta correctas, los códigos de las reglas y el motivo. Si corresponde, el saldo del fondo actualizado y el registro del veredicto.

**Comprobación.** Un peaje con auto propio declarado en Viajes Internos (P02) debe quedar observado y reclasificado a P01 / Peajes / CTA-5100, citando RG-09. El criterio general es el golden set de 34 casos: el evaluador los corre con Gemini real, y la corrida final debe pasar todos.

**Por qué un agente.** El LLM interpreta la glosa y la contrasta con la política. Las reglas exactas se calculan con código, para que sean auditables. El ciclo ReAct permite buscar, validar, corregir y decidir, o detenerse de forma segura.

Detalle completo en [docs/caso_y_criterio.md](docs/caso_y_criterio.md).

## 2. Entorno y carga de configuración

In [ ]:
import os

from src.config import CARPETA_PROYECTO, obtener_llm, registrar_traza
from src.contrato import MAX_PASOS, DECISIONES

# src.config carga el .env del proyecto; mostramos presencia, nunca secretos.
print('Proyecto:', CARPETA_PROYECTO)
print('GEMINI_MODEL:', os.getenv('GEMINI_MODEL') or 'FALTA')
print('EMBEDDING_MODEL:', os.getenv('EMBEDDING_MODEL') or 'FALTA')
print('GOOGLE_API_KEY:', 'configurada (valor oculto)' if os.getenv('GOOGLE_API_KEY', '').strip() else 'FALTA')
print('REDIS_URL:', 'configurada (valor oculto)' if os.getenv('REDIS_URL', '').strip() else 'FALTA')
print('REDIS_PREFIJO:', os.getenv('REDIS_PREFIJO') or 'FALTA')
print('MOSTRAR_TRAZAS:', os.getenv('MOSTRAR_TRAZAS', '1'))
print('MAX_PASOS:', MAX_PASOS, '| decisiones:', ', '.join(DECISIONES))

## 3. Caso normal: ReAct con herramienta, observación, decisión y parada

La traza debe mostrar la búsqueda, su observación, la validación, su observación y la respuesta final.

In [ ]:
import csv
import json

from src.config import CARPETA_PROYECTO
from src.grafo import construir_agente, conversar_con_caso

# G-001 es ficticio y sirve para recorrer el ciclo ReAct completo.
with (CARPETA_PROYECTO / 'datos' / 'rendiciones_prueba.csv').open(encoding='utf-8', newline='') as archivo:
    gastos = list(csv.DictReader(archivo))
gasto_normal = next(gasto for gasto in gastos if gasto['id'] == 'G-001')
gasto_normal['monto'] = int(gasto_normal['monto'])
gasto_normal['vb_gerente_general'] = gasto_normal['vb_gerente_general'].strip().casefold() == 'true'
for campo in ('patente', 'numero_ot'):
    gasto_normal[campo] = gasto_normal[campo] or None

agente_normal = construir_agente()
hilo_normal = 'DEMO-REACT-G001'
mensaje_normal = 'Revisa este gasto rendido: ' + json.dumps(gasto_normal, ensure_ascii=False)
respuesta_normal = conversar_con_caso(agente_normal, mensaje_normal, hilo_normal, gasto_normal)
print('Respuesta final:', respuesta_normal)
assert respuesta_normal.strip(), 'El agente debe entregar una respuesta.'

# Muestra el estado estructurado que el grafo conservó en este hilo.
estado_normal = agente_normal.get_state({'configurable': {'thread_id': hilo_normal}}).values
for campo in ('gasto', 'clasificacion', 'validacion', 'veredicto'):
    print(f'{campo}: {estado_normal.get(campo)}')

## 4. Segundo turno: uso del historial

In [ ]:
from src.grafo import construir_agente, conversar

# Un agente conserva el historial por thread_id; cada caso usa un hilo distinto.
agente_historial = construir_agente()

# GS-31: un saludo breve debe responderse sin herramientas.
respuesta_gs31 = conversar(agente_historial, 'Hola', 'GS-31')
print('GS-31:', respuesta_gs31)

# GS-32: los dos turnos usan exactamente el mismo hilo.
hilo_gs32 = 'GS-32'
respuesta_turno_1 = conversar(agente_historial, 'Soy Pedro, chofer del Centro de Distribución', hilo_gs32)
respuesta_turno_2 = conversar(agente_historial, '¿Cómo me llamo y en qué unidad trabajo?', hilo_gs32)
print('GS-32 turno 1:', respuesta_turno_1)
print('GS-32 turno 2:', respuesta_turno_2)

assert 'pedro' in respuesta_turno_2.casefold(), 'El segundo turno debe recordar el nombre.'
assert any(texto in respuesta_turno_2.casefold() for texto in ('centro de distribución', 'centro de distribucion', 'cd')), 'El segundo turno debe recordar la unidad.'
# En las trazas de ambos turnos debe verse pide_herramientas vacío.

## 5. Seguridad básica: fuera de alcance y jailbreak

In [ ]:
from src.seguridad import PROMPT_SISTEMA

# TODO(conocimiento-seguridad): pregunta fuera de alcance, jailbreak directo y jailbreak en la glosa

## 6. Ampliación RAG (Redis)

In [ ]:
from src.rag import buscar_politica

# TODO(conocimiento-seguridad): búsqueda con fragmentos y un caso donde el agente NO busca

## 7. Ampliación memoria del fondo (GS-26)

In [ ]:
import json
import tempfile
from pathlib import Path

from src import registro
from src.config import CARPETA_PROYECTO
from src.grafo import construir_agente, conversar_con_caso
from src.memoria import fondo_inicial, resumen_fondo

# GS-26 usa dos gastos del mismo fondo y el mismo hilo. Son solo dos turnos Gemini.
golden_memoria = json.loads((CARPETA_PROYECTO / 'pruebas' / 'golden_set.json').read_text(encoding='utf-8'))
caso_gs26 = next(caso for caso in golden_memoria['casos'] if caso['id'] == 'GS-26')
contexto_fondo = caso_gs26['contexto']['fondo']
fondo = fondo_inicial(contexto_fondo['id_fondo'])
assert fondo['saldo_inicial'] == contexto_fondo['saldo_inicial']
agente = construir_agente()
hilo_gs26 = 'notebook-gs26-memoria'
decisiones_gs26 = []

# El demo no modifica el registro local del usuario.
ruta_registro_original = registro.RUTA_REGISTRO
with tempfile.TemporaryDirectory() as carpeta_temporal:
    try:
        registro.RUTA_REGISTRO = Path(carpeta_temporal) / 'registro_gs26.db'
        for numero_turno, turno in enumerate(caso_gs26['turnos'], start=1):
            gasto = turno['gasto']
            mensaje = turno['mensaje'] + '\nGasto rendido:\n' + json.dumps(gasto, ensure_ascii=False)
            respuesta = conversar_con_caso(agente, mensaje, hilo_gs26, gasto, fondo)
            estado = agente.get_state({'configurable': {'thread_id': hilo_gs26}}).values
            veredicto = estado.get('veredicto')
            assert veredicto, f"Turno {numero_turno}: faltó un veredicto estructurado."
            fondo = estado['fondo']
            decisiones_gs26.append(veredicto['decision'])
            print(f"Turno {numero_turno}: {gasto['id']} -> {veredicto['decision'].upper()} | saldo {fondo['saldo_actual']:,} CLP")
            print(respuesta)
    finally:
        registro.RUTA_REGISTRO = ruta_registro_original

print(resumen_fondo(fondo))
assert decisiones_gs26 == ['aprobar', 'observar']
assert fondo['saldo_actual'] == 20000
assert fondo['gastos_aprobados'] == ['G-2601']

## 8. Ampliación acción: registro de veredictos

In [ ]:
import json
import tempfile
from pathlib import Path

from src import registro
from src.config import CARPETA_PROYECTO
from src.grafo import construir_agente, conversar_con_caso
from src.memoria import fondo_inicial

# La escritura ocurre al cerrar una corrida real del agente; usamos SQLite temporal.
golden_registro = json.loads((CARPETA_PROYECTO / 'pruebas' / 'golden_set.json').read_text(encoding='utf-8'))
caso_registro = next(caso for caso in golden_registro['casos'] if caso['id'] == 'GS-07')
turno_registro = caso_registro['turnos'][0]
gasto_registro = turno_registro['gasto']
mensaje_registro = turno_registro['mensaje'] + '\nGasto rendido:\n' + json.dumps(gasto_registro, ensure_ascii=False)
ruta_registro_original = registro.RUTA_REGISTRO
with tempfile.TemporaryDirectory() as carpeta_temporal:
    try:
        registro.RUTA_REGISTRO = Path(carpeta_temporal) / 'registro_agente.db'
        antes_registro = registro.leer_registro()
        agente_registro = construir_agente()
        respuesta_registro = conversar_con_caso(
            agente_registro, mensaje_registro, 'notebook-registro-gs07', gasto_registro,
            fondo_inicial(gasto_registro['id_fondo']),
        )
        estado_registro = agente_registro.get_state(
            {'configurable': {'thread_id': 'notebook-registro-gs07'}}
        ).values
        despues_registro = registro.leer_registro()
        print('Registro antes:', antes_registro)
        print('Respuesta del agente:', respuesta_registro)
        print('Veredicto estructurado:', estado_registro.get('veredicto'))
        print('Registro después:', despues_registro)
        assert estado_registro.get('veredicto'), 'La corrida no produjo un veredicto estructurado.'
        assert len(despues_registro) == 1
        assert despues_registro[0]['id_gasto'] == gasto_registro['id']
        assert despues_registro[0]['decision'] == estado_registro['veredicto']['decision']
        print('El nodo de cierre registró el resultado en la base temporal; no se llamó registrar_veredicto desde el notebook.')
    finally:
        registro.RUTA_REGISTRO = ruta_registro_original

## 9. Ampliación juez

In [ ]:
from src.seguridad import juzgar

# TODO(conocimiento-seguridad): modelo, prompt y veredictos del juez visibles

## 10. Golden set: ejecución y resultados por caso

In [ ]:
from pruebas.evaluar import evaluar, mostrar_tabla

# Muestra acotada: no ejecutes el golden completo desde el notebook.
CASOS_MUESTRA = ['GS-07']
resultados_muestra = evaluar(casos=CASOS_MUESTRA, pausa=0)
mostrar_tabla(resultados_muestra)

## 11. Verificación final

Revisión del contrato (no necesita clave de API).

In [ ]:
import subprocess
import sys

# Usa el mismo Python del notebook (su entorno virtual).
resultado = subprocess.run([sys.executable, 'verificar.py'], capture_output=True, text=True, encoding='utf-8')
print(resultado.stdout, resultado.stderr)